# Biohub DeepCenter Missing-19 Recovery

This notebook restores the missing 19 DeepCenter prediction `.geff` folders and copies them into the full-199 checkpoint directory.

It does **not** rerun all 199 videos. It only runs the 19 missing `6bba_*` samples.

## 1. Mount Drive and Check Paths

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount note:', repr(e))

PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
BASE_DIR = PROJECT_DIR / 'data'
REPORT_DIR = PROJECT_DIR / 'reports'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
SUPPORT_DIR = ARTIFACT_DIR / 'biohub-tracking-support-pack-50ep-v1'
CHECKPOINT_DIR = REPORT_DIR / 'deepcenter_full199_prediction_geff_checkpoint'

WORK_DIR = Path('/content/biohub_missing19_work')
REPO_DIR = WORK_DIR / 'tracking_repo'
LOCAL_TEST_DIR = Path('/content/biohub_deepcenter_validation/test')

missing_ids = [
    '6bba_df673a83', '6bba_e16ffc58', '6bba_e5e44988', '6bba_ebdf3b34',
    '6bba_ebff6e76', '6bba_ed9377fd', '6bba_edf14583', '6bba_eebc57a5',
    '6bba_ef7b4f7e', '6bba_f17befbc', '6bba_f1fde7e0', '6bba_f20478e9',
    '6bba_f4ae811c', '6bba_f8ffd5e7', '6bba_fbc898dc', '6bba_fc516dc6',
    '6bba_fc5f39dc', '6bba_fc83837d', '6bba_fe670320'
]

print('PROJECT_DIR:', PROJECT_DIR, PROJECT_DIR.exists())
print('BASE_DIR:', BASE_DIR, BASE_DIR.exists())
print('SUPPORT_DIR:', SUPPORT_DIR, SUPPORT_DIR.exists())
print('CHECKPOINT_DIR:', CHECKPOINT_DIR, CHECKPOINT_DIR.exists())
print('checkpoint geff before:', len(list(CHECKPOINT_DIR.glob('*.geff'))) if CHECKPOINT_DIR.exists() else 0)
print('missing ids:', len(missing_ids))

assert PROJECT_DIR.exists(), 'Drive project folder not found. Remount Drive or fix PROJECT_DIR.'
assert (BASE_DIR / 'train').exists(), 'data/train not found.'
assert SUPPORT_DIR.exists(), 'Support artifact not extracted. Run artifact extraction first.'
assert (SUPPORT_DIR / 'repo/scripts/predict_unet_transformer.py').exists(), 'Support repo script missing.'

## 2. Install Offline Dependencies From Support Pack

In [ ]:
import importlib

WHEEL_DIR = SUPPORT_DIR / 'wheels'
print('WHEEL_DIR:', WHEEL_DIR, WHEEL_DIR.exists())

def missing_imports(import_names):
    missing = []
    for name in import_names:
        try:
            importlib.import_module(name)
        except Exception:
            missing.append(name)
    return missing

required_imports = [
    'zarr', 'tracksdata', 'geff', 'pyscipopt', 'ilpy', 'imagecodecs',
    'rustworkx', 'numcodecs', 'donfig', 'bidict', 'deprecated', 'polars'
]

missing_before = missing_imports(required_imports)
print('missing before:', missing_before)

if missing_before:
    assert WHEEL_DIR.exists(), 'Offline wheels folder missing.'
    packages = [
        'polars', 'tracksdata', 'zarr', 'pyscipopt', 'geff', 'geff_spec',
        'ilpy', 'imagecodecs', 'rustworkx', 'numcodecs', 'donfig', 'bidict', 'deprecated'
    ]
    cmd = [
        sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
        '--find-links', str(WHEEL_DIR), *packages
    ]
    print('install command:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

missing_after = missing_imports(required_imports)
print('missing after:', missing_after)

# Colab's polars build may not expose polars.Float16, which makes tracksdata import fail.
# The prediction subprocess is patched later with sitecustomize.py, so this is non-blocking here.
allowed_missing = {'tracksdata'}
blocking_missing = [m for m in missing_after if m not in allowed_missing]
print('allowed missing:', sorted(set(missing_after) & allowed_missing))
print('blocking missing:', blocking_missing)
assert not blocking_missing, f'Still missing blocking imports: {blocking_missing}'

## 3. Build Pseudo-Test Folder for Only the Missing 19

In [ ]:
LOCAL_TEST_DIR.mkdir(parents=True, exist_ok=True)

for item in LOCAL_TEST_DIR.glob('*'):
    if item.is_symlink() or item.is_file():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

missing_data = []
for sample_id in missing_ids:
    src = BASE_DIR / 'train' / f'{sample_id}.zarr'
    dst = LOCAL_TEST_DIR / f'{sample_id}.zarr'
    if not src.exists():
        missing_data.append(str(src))
        continue
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)

print('missing_data:', len(missing_data))
for p in missing_data:
    print(p)
assert not missing_data, 'Some missing-19 zarr files are not available in data/train.'

print('recovery pseudo-test zarr:', len(list(LOCAL_TEST_DIR.glob('*.zarr'))))
print([p.name for p in sorted(LOCAL_TEST_DIR.glob('*.zarr'))[:5]], '...')

## 4. Prepare Inference Repo and Split File

In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_SRC = SUPPORT_DIR / 'repo'

if REPO_DIR.exists():
    print('repo already exists:', REPO_DIR)
else:
    print('copy repo:', REPO_SRC, '->', REPO_DIR)
    shutil.copytree(REPO_SRC, REPO_DIR)

pred_dir = REPO_DIR / 'predictions/unknown/unet_transformer/split_0'
if pred_dir.exists():
    shutil.rmtree(pred_dir)
pred_dir.mkdir(parents=True, exist_ok=True)

split_path = REPO_DIR / 'kaggle_test_splits_50ep.json'
split_path.write_text(json.dumps({'0': missing_ids}, indent=2))

print('REPO_DIR:', REPO_DIR)
print('predict script:', (REPO_DIR / 'scripts/predict_unet_transformer.py').exists())
print('weights:', (REPO_DIR / 'weights/unet_transformer/split_0/edge_predictor_best.pth').exists())
sitecustomize_code = """
try:
    import polars as pl
    if not hasattr(pl, 'Float16'):
        pl.Float16 = pl.Float32
except Exception:
    pass
""".strip() + "\n"
for patch_path in [REPO_DIR / 'sitecustomize.py', REPO_DIR / 'src/sitecustomize.py']:
    patch_path.parent.mkdir(parents=True, exist_ok=True)
    patch_path.write_text(sitecustomize_code)
    print('wrote compatibility patch:', patch_path)

print('split_path:', split_path)
print('split sample count:', len(missing_ids))

assert (REPO_DIR / 'scripts/predict_unet_transformer.py').exists()
assert (REPO_DIR / 'weights/unet_transformer/split_0/edge_predictor_best.pth').exists()

## 5. Run DeepCenter Prediction for the Missing 19

In [ ]:
predict_cmd = [
    sys.executable,
    'scripts/predict_unet_transformer.py',
    '--data-dir', str(LOCAL_TEST_DIR),
    '--splits', 'kaggle_test_splits_50ep.json',
    '--split', '0',
    '--weights', 'weights/unet_transformer/split_0/edge_predictor_best.pth',
    '--unet-batch-size', '4',
    '--det-threshold', '0.99',
    '--ilp-edge-weight', '-1.0',
    '--ilp-appearance-weight', '0.1',
    '--ilp-disappearance-weight', '0.1',
    '--ilp-division-weight', '1.0',
    '--use-ilp',
]

print('test videos:', len(list(LOCAL_TEST_DIR.glob('*.zarr'))))
print('command:')
print(' '.join(predict_cmd))

start = time.time()
pythonpath = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / 'src'), 'src'])
print('PYTHONPATH:', pythonpath)

result = subprocess.run(
    predict_cmd,
    cwd=REPO_DIR,
    env={**os.environ, 'PYTHONPATH': pythonpath},
    text=True,
    capture_output=True,
)

print('RETURN CODE:', result.returncode)
print('\n--- STDOUT tail ---')
print(result.stdout[-10000:])
print('\n--- STDERR tail ---')
print(result.stderr[-10000:])
print(f'elapsed minutes: {(time.time() - start) / 60:.2f}')

if result.returncode != 0:
    raise RuntimeError('Prediction failed; see STDOUT/STDERR above.')

pred_dir = REPO_DIR / 'predictions/unknown/unet_transformer/split_0'
print('pred_dir:', pred_dir)
print('pred geff:', len(list(pred_dir.glob('*.geff'))))
assert len(list(pred_dir.glob('*.geff'))) == len(missing_ids), 'Prediction count is not 19.'

## 6. Copy Recovered Predictions Into the Full-199 Checkpoint

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
pred_dir = REPO_DIR / 'predictions/unknown/unet_transformer/split_0'

copied = []
not_found = []

for sample_id in missing_ids:
    src = pred_dir / f'{sample_id}.geff'
    if not src.exists():
        not_found.append(sample_id)
        continue

    dst = CHECKPOINT_DIR / f'{sample_id}.geff'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    copied.append(sample_id)

checkpoint_count = len(list(CHECKPOINT_DIR.glob('*.geff')))
print('copied:', len(copied), copied)
print('not_found:', len(not_found), not_found)
print('checkpoint now:', checkpoint_count)

assert not not_found, 'Some recovered predictions were not copied.'
assert checkpoint_count == 199, 'Checkpoint still incomplete.'
print('CHECKPOINT OK: 199/199. Now run Biohub_DeepCenter_Postprocess_Tuning.ipynb.')

## 7. Optional Final Missing Check

In [ ]:
expected_ids = sorted(
    {p.name[:-5] for p in (BASE_DIR / 'train').glob('*.zarr')}
    & {p.name[:-5] for p in (BASE_DIR / 'train').glob('*.geff')}
)
checkpoint_ids = sorted(p.name[:-5] for p in CHECKPOINT_DIR.glob('*.geff'))
still_missing = sorted(set(expected_ids) - set(checkpoint_ids))

print('expected:', len(expected_ids))
print('checkpoint:', len(checkpoint_ids))
print('still_missing:', len(still_missing), still_missing)